# Featurizer Tutorial: Basic Aggregations

This notebook demonstrates the core workflow of Featurizer using a simple e-commerce scenario:
- **Customers** (parent entity) with attributes like country and age
- **Orders** (child entity) with amounts and statuses

We'll generate features for customers by aggregating their order history.

## 1. Setup

First, let's set up the environment and create sample data.

In [ ]:
import sys
from pathlib import Path

# Add featurizer to path
sys.path.insert(0, str(Path.cwd().parent.parent))

# Create sample data if it doesn't exist
if not Path("data.db").exists():
    exec(open("create_data.py").read())

## 2. Understanding the Configuration

Featurizer uses YAML configuration to define entities, relationships, and feature generation parameters.

In [ ]:
# Let's examine the configuration file
with open("config.yaml") as f:
    print(f.read())

### Key Configuration Elements:

- **target**: The entity we want to generate features for (`customers`)
- **max_depth**: How deep to traverse relationships (2 levels)
- **intervals**: Time windows for aggregations (`P7D` = 7 days, `P30D` = 30 days)
- **entities**: Define tables with their columns and types
- **relationships**: Define parent-child connections via foreign keys

## 3. Creating the Featurizer

Load the configuration and create a Featurizer instance.

In [ ]:
from featurizer import Featurizer

# Create featurizer from config
featurizer = Featurizer("config.yaml")

print(f"Target entity: {featurizer.target.alias}")
print(f"Max depth: {featurizer.max_depth}")
print(f"Intervals: {featurizer.intervals}")
print(f"Number of entities: {len(list(featurizer.entities))}")
print(f"Number of relationships: {len(featurizer.relationships)}")

## 4. Exploring the Entity Graph

Let's examine the entities and their relationships.

In [ ]:
# List all entities
print("Entities:")
for entity in featurizer.entities:
    print(f"\n  {entity.alias}:")
    print(f"    Table: {entity.table}")
    print(f"    ID: {entity.id.name if entity.id else 'None'}")
    print(
        f"    Temporal index: {entity.temporal_ix.name if entity.temporal_ix else 'None'}"
    )
    print(f"    Features: {len(entity.features)}")

In [ ]:
# Show relationships
print("Relationships:")
for rel in featurizer.relationships:
    print(f"  {rel.parent.alias}.{rel.parent_key} -> {rel.child.alias}.{rel.child_key}")

## 5. Generated Features

Featurizer automatically synthesizes features by:
1. Taking base features from each entity (columns declared as variables)
2. Applying aggregations when traversing backward relationships
3. Applying transformations to all features

In [ ]:
# Get features for the target entity
target_features = featurizer.features[featurizer.target.alias]

print(f"Total features generated: {len(target_features)}")
print("\nSample features (first 20):")
for i, feature in enumerate(sorted(target_features, key=lambda f: f.name)[:20], 1):
    print(f"  {i:2}. {feature.name}")

In [ ]:
# Categorize features by type
feature_types = {}
for f in target_features:
    feature_types.setdefault(f.type, []).append(f)

print("Features by type:")
for ftype, features in sorted(feature_types.items()):
    print(f"  {ftype}: {len(features)}")

## 6. Understanding Feature Names

Feature names follow a pattern that describes how they were created:

- `AGGREGATION(entity.column)` - Basic aggregation
- `AGGREGATION(entity.column|interval=P7D)` - Time-windowed aggregation
- `TRANSFORM(entity.column)` - Transformation applied

In [ ]:
# Show aggregation features
agg_features = [
    f
    for f in target_features
    if "SUM(" in f.name or "MEAN(" in f.name or "COUNT(" in f.name
]

print("Sample aggregation features:")
for f in sorted(agg_features, key=lambda x: x.name)[:10]:
    print(f"  {f.name}")

In [ ]:
# Show time-windowed features
windowed = [f for f in target_features if "interval=" in f.name]

print(f"Time-windowed features: {len(windowed)}")
print("\nSample windowed features:")
for f in sorted(windowed, key=lambda x: x.name)[:10]:
    print(f"  {f.name}")

## 7. Examining the Generated SQL

Featurizer generates a PostgreSQL query with CTEs (Common Table Expressions) for each stage.

In [ ]:
# Get the generated SQL query
sql = featurizer.query

print("Generated SQL Query:")
print("=" * 80)
print(sql)
print("=" * 80)

### SQL Structure:

The query has this structure:
```sql
SELECT aod.as_of_date, t.*
FROM as_of_dates AS aod
CROSS JOIN LATERAL (
    WITH
        -- CTEs for each entity's synthesis and transformation
        orders_synth AS (...),
        orders_transform AS (...),
        orders_aggs_for_customers AS (...),
        customers_synth AS (...),
        customers_transform AS (...)
    SELECT * FROM customers_transform
) AS t
ORDER BY aod.as_of_date
```

In [ ]:
# Show the CTEs that were generated
print(f"Number of CTEs: {len(featurizer.ctes)}")
print("\nCTE names:")
for cte in featurizer.ctes:
    # Extract CTE name from the query
    lines = cte.strip().split("\n")
    for line in lines:
        if " as (" in line:
            name = line.split(" as (")[0].strip().lstrip("-").strip()
            print(f"  - {name}")
            break

## 8. Available Primitives

Featurizer comes with many built-in primitives. Let's explore them.

In [ ]:
from featurizer.primitives.utils import list_aggregations, list_transformations

# List available aggregations
aggs = list(list_aggregations())
print(f"Available aggregations ({len(aggs)}):")
print(f"  {', '.join(aggs)}")

In [ ]:
# List available transformations
transforms = list(list_transformations())
print(f"\nAvailable transformations ({len(transforms)}):")

# Group by category
categories = {
    "math": [
        "abs",
        "exp",
        "ln",
        "log",
        "sqrt",
        "cbrt",
        "sign",
        "ceil",
        "floor",
        "trunc",
    ],
    "date": [
        "day",
        "dow",
        "dom",
        "doy",
        "year",
        "month",
        "hour",
        "quarter",
        "week",
        "week_of_year",
        "century",
    ],
    "rolling": [t for t in transforms if "rolling" in t],
    "lag": [t for t in transforms if "lag_" in t],
    "cumulative": ["cum_sum", "cum_mean", "cum_max", "cum_min", "cum_count"],
}

for cat, items in categories.items():
    available = [i for i in items if i in transforms]
    if available:
        print(f"  {cat}: {', '.join(available)}")

## 9. Summary

In this tutorial, we learned:

1. **Configuration**: How to define entities, relationships, and feature generation parameters in YAML
2. **Entity Graph**: How Featurizer models the relationships between tables
3. **Feature Synthesis**: How features are automatically generated through aggregations and transformations
4. **SQL Generation**: How the planner produces CTEs and the renderer creates the final query
5. **Primitives**: The available aggregation and transformation primitives

### Next Steps:
- Try Example 2 (Temporal Joins) for as-of join semantics
- Try Example 3 (Deep Nesting) for multi-level relationships
- Try Example 4 (Custom Primitives) to extend Featurizer

In [ ]:
# Final summary
print("Feature Generation Summary")
print("=" * 40)
print(f"Target: {featurizer.target.alias}")
print(f"Depth: {featurizer.max_depth}")
print(f"Intervals: {', '.join(featurizer.intervals)}")
print(f"Total features: {len(target_features)}")
print(f"SQL query length: {len(sql):,} characters")
print(f"CTEs generated: {len(featurizer.ctes)}")